In [1]:
import sys, os
ROOT = "/Users/fserracrespi/Desktop/PD_PROJECT_UOFL" 
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from utils.Decode import decoder_DF
from utils.dataframe_cols import cols_asignacion2
from utils.Prog_df import check_progression
from utils.Prog_df import progression_csv_upgrade
from utils.Prog_df import check_progression_multi
from utils.Prog_df import progression_multi_csv_upgrade
from utils.Prog_df import visit_csv_upgrade
from utils.Prog_df import check_visits
import json
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kurtosis, skew
import re
import math

In [2]:
path2='/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Data_Dictionary_-_Harmonized_12Sep2025.csv'
path3='/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Code_List_-_Harmonized_12Sep2025.csv'
orden_visitas = ["BL", "V04", "V06", "V08", "V10", "V12"]
code_cols=pd.read_csv(path2, dtype=str)
code_rows=pd.read_csv(path3, dtype=str)
PATNOs = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Relevant_presnece_PATNO', dtype=str)['PATNO'].tolist()
print(f'Total PATNOs to process: {len(PATNOs)}')

Total PATNOs to process: 1056


# Biospecimen data

In [3]:
blood_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Biospecimen/Lab_Collection_Procedures/DATA/Blood_Chemistry___Hematology_12Sep2025.csv',dtype=str)
blood_data=blood_data.loc[blood_data['PATNO'].isin(PATNOs)]
blood_data=decoder_DF(blood_data, code_rows, code_cols, module='COVANCE')
blood_data=blood_data.loc[blood_data['Visit ID'].isin(orden_visitas)]
blood_data['Visit ID'] = pd.Categorical(blood_data['Visit ID'], categories=orden_visitas, ordered=True)
blood_data=blood_data.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)     
blood_data

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,PATNO,Visit ID,Page Name,Collection Date,Collection Time,Date Received,Time Received,Report Date,Report Time,Lab Code,...,SI Result,SI Unit,SI Low Range,SI High Range,US Result,US Unit,US Low Range,US High Range,Result Flag,Test Specific Comments
0,100891,BL,COVANCE,04/2021,12:20:00,04/2021,04:24:00,04/2021,10:13:00,COVANCE,...,393,umol/L,149,494,6.6,mg/dL,2.5,8.3,NaN,NaN
1,100891,BL,COVANCE,04/2021,12:20:00,04/2021,04:24:00,04/2021,10:13:00,COVANCE,...,7.1,mmol/L,1.4,8.6,20,mg/dL,4,24,NaN,NaN
2,100891,BL,COVANCE,04/2021,12:20:00,04/2021,04:24:00,04/2021,10:13:00,COVANCE,...,25,U/L,8,40,25,U/L,8,40,NaN,NaN
3,100891,BL,COVANCE,04/2021,12:20:00,04/2021,04:24:00,04/2021,10:30:00,COVANCE,...,45,U/L,40,129,45,U/L,40,129,NaN,NaN
4,100891,BL,COVANCE,04/2021,12:20:00,04/2021,04:24:00,04/2021,10:13:00,COVANCE,...,95,umol/L,40,119,1.08,mg/dL,0.45,1.35,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41643,75562,V04,COVANCE,01/2020,09:22:00,01/2020,04:48:00,01/2020,09:48:00,COVANCE,...,244,umol/L,149,446,4.1,mg/dL,2.5,7.5,NaN,NaN
41644,75562,V04,COVANCE,01/2020,09:22:00,01/2020,04:48:00,01/2020,09:48:00,COVANCE,...,5,umol/L,3,21,0.3,mg/dL,0.2,1.2,NaN,NaN
41645,75562,V04,COVANCE,01/2020,09:22:00,01/2020,04:48:00,01/2020,09:48:00,COVANCE,...,79,g/L,60,80,7.9,g/dL,6.0,8.0,NaN,NaN
41646,75562,V04,COVANCE,01/2020,09:22:00,01/2020,04:48:00,01/2020,09:48:00,COVANCE,...,5.7,mmol/L,1.4,8.6,16,mg/dL,4,24,NaN,NaN


In [4]:
# --- Función para rellenar valores faltantes ---
def fill_miss_text(lista_val):
    lista_val = lista_val.copy()
    # Rellenar hacia adelante
    for i in range(1, len(lista_val)):
        if pd.isna(lista_val[i]):
            lista_val[i] = lista_val[i-1]
    # Rellenar hacia atrás
    for i in range(len(lista_val)-2, -1, -1):
        if pd.isna(lista_val[i]):
            lista_val[i] = lista_val[i+1]
    return lista_val


dict_problematic_PATNOs = {}
for id in blood_data['PATNO'].unique():
    data_patient = blood_data.loc[blood_data['PATNO'] == id,:]
    if data_patient['SI Result'].isna().any():
        dict_problematic_PATNOs[id] = data_patient.loc[data_patient['SI Result'].isna(),'Test Name'].unique().tolist()



for patno in dict_problematic_PATNOs:
    mask = blood_data['PATNO'] == patno
    for test in dict_problematic_PATNOs[patno]:
        valores_originales = blood_data.loc[mask & (blood_data['Test Name'] == test), 'SI Result'].tolist()
        valores_rellenos = fill_miss_text([v if pd.notna(v) else np.nan for v in valores_originales])
        blood_data.loc[mask & (blood_data['Test Name'] == test), 'SI Result'] = valores_rellenos

blood_data.isna().sum()

PATNO                         0
Visit ID                      0
Page Name                     0
Collection Date               0
Collection Time               0
Date Received                 0
Time Received                 0
Report Date                 496
Report Time                 496
Lab Code                      0
Lab Group                     0
Test Code                     0
Test Name                     0
Visit Label                  95
SI Result                   229
SI Unit                    2670
SI Low Range               2097
SI High Range              2097
US Result                   788
US Unit                    1337
US Low Range               2097
US High Range              2097
Result Flag               40133
Test Specific Comments    40344
dtype: int64

In [5]:
def SI_result(row):
    if pd.isna(row['SI Result']):
        return row['Test Specific Comments']
    else:
        return row['SI Result']

blood_data['SI Result'] = blood_data.apply(SI_result, axis=1)
relevant_cols = ['PATNO', 'Visit ID', 'Test Name', 'SI Result']
blood_data= blood_data[relevant_cols]
blood_data

,PATNO,Visit ID,Test Name,SI Result
0,100891,BL,Serum Uric Acid,393
1,100891,BL,Urea Nitrogen,7.1
2,100891,BL,AST (SGOT),25
3,100891,BL,Alkaline Phosphatase-QT,45
4,100891,BL,Creatinine(Rate Blanked)-2dp,95
...,...,...,...,...
41643,75562,V04,Serum Uric Acid,244
41644,75562,V04,Total Bilirubin,5
41645,75562,V04,Total Protein,79
41646,75562,V04,Urea Nitrogen,5.7


In [6]:
blood_data.duplicated(subset=['PATNO', 'Visit ID', 'Test Name'])
blood_data.loc[blood_data.duplicated(subset=['PATNO', 'Visit ID', 'Test Name']),:]

blood_data = blood_data.drop_duplicates(subset=['PATNO', 'Visit ID', 'Test Name'], keep='first').reset_index(drop=True)

blood_data = blood_data.pivot_table(
    index=['PATNO', 'Visit ID'], 
    columns='Test Name', 
    values='SI Result',
    aggfunc=lambda x: ' '.join(x.astype(str))  # conserva el texto
).reset_index()

blood_data.columns.name = None
blood_data

/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_61418/2057543336.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  blood_data = blood_data.pivot_table(


,PATNO,Visit ID,ALT (SGPT),APTT-FSL-QT,APTT-QT,AST (SGOT),Albumin-BCG,Albumin-QT,Alkaline Phosphatase-QT,Bands,...,Serum Glucose,Serum Potassium,Serum Sodium,Serum Uric Acid,"Serum beta hCG, Qualitative-QT",Total Bilirubin,Total Protein,Urea Nitrogen,WBC,Which visit being performed?
0,100891,BL,25,25.7,NaN,25,45,NaN,45,NaN,...,3.8,4.6,139,393,NaN,7,66,7.1,6.66,NaN
1,101295,BL,26,26.9,NaN,23,45,NaN,61,NaN,...,9.2,4.2,142,208,NaN,7,62,5.0,4.95,NaN
2,101482,BL,20,No specimen received,NaN,21,46,NaN,82,NaN,...,5.1,4.5,140,315,NaN,14,68,10.0,7.25,NaN
3,101516,BL,20,22.0,NaN,26,52,NaN,73,NaN,...,5.6,4.3,138,345,NaN,15,79,9.3,5.39,NaN
4,101749,BL,31,21.3,NaN,26,50,NaN,85,NaN,...,5.3,4.6,138,482,NaN,7,79,6.8,Clotted Blood Received,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1328,75480,V04,23,NaN,NaN,13,NaN,44,86,NaN,...,8.3,4.2,138,291,NaN,19,78,8.9,9.24,NaN
1329,75484,V04,18,NaN,NaN,23,NaN,47,45,NaN,...,4.7,4.5,143,256,NaN,14,73,4.3,4.62,NaN
1330,75505,V04,7,NaN,NaN,12,NaN,41,91,NaN,...,4.7,4.2,139,315,NaN,5,65,6.1,4.89,NaN
1331,75524,V04,27,NaN,NaN,23,NaN,44,77,NaN,...,4.7,3.9,138,297,NaN,7,69,5.7,4.28,NaN


In [7]:
blood_data.isna().sum()
blood_data = blood_data.dropna(axis=1, how='any')
blood_data.drop(columns=blood_data.filter(like='(%)').columns.tolist(), inplace=True)
blood_data.isna().sum()



/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_61418/2942987204.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blood_data.drop(columns=blood_data.filter(like='(%)').columns.tolist(), inplace=True)


PATNO                      0
Visit ID                   0
ALT (SGPT)                 0
AST (SGOT)                 0
Alkaline Phosphatase-QT    0
Basophils                  0
Calcium (EDTA)             0
Eosinophils                0
Hematocrit                 0
Hemoglobin                 0
Lymphocytes                0
Monocytes                  0
Neutrophils                0
Platelets                  0
RBC                        0
RBC Morphology             0
Serum Bicarbonate          0
Serum Chloride             0
Serum Glucose              0
Serum Potassium            0
Serum Sodium               0
Serum Uric Acid            0
Total Bilirubin            0
Total Protein              0
Urea Nitrogen              0
WBC                        0
dtype: int64

In [8]:
blood_data[blood_data.columns[2:]] = blood_data[blood_data.columns[2:]].apply(pd.to_numeric, errors='coerce')
blood_data[blood_data.columns[2:]] = blood_data[blood_data.columns[2:]].fillna(blood_data[blood_data.columns[2:]].mean())
blood_data.drop(columns=['RBC Morphology'], inplace=True)
blood_data.isna().sum()



/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_61418/26092149.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blood_data[blood_data.columns[2:]] = blood_data[blood_data.columns[2:]].apply(pd.to_numeric, errors='coerce')
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_61418/26092149.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  blood_data[blood_data.columns[2:]] = blood_data[blood_data.columns[2:]].fillna(blood_data[blood_data.columns[2:]].mean())
/var/folders/vr/_v9wq92941bf

PATNO                      0
Visit ID                   0
ALT (SGPT)                 0
AST (SGOT)                 0
Alkaline Phosphatase-QT    0
Basophils                  0
Calcium (EDTA)             0
Eosinophils                0
Hematocrit                 0
Hemoglobin                 0
Lymphocytes                0
Monocytes                  0
Neutrophils                0
Platelets                  0
RBC                        0
Serum Bicarbonate          0
Serum Chloride             0
Serum Glucose              0
Serum Potassium            0
Serum Sodium               0
Serum Uric Acid            0
Total Bilirubin            0
Total Protein              0
Urea Nitrogen              0
WBC                        0
dtype: int64

In [9]:

blood_data


,PATNO,Visit ID,ALT (SGPT),AST (SGOT),Alkaline Phosphatase-QT,Basophils,Calcium (EDTA),Eosinophils,Hematocrit,Hemoglobin,...,Serum Bicarbonate,Serum Chloride,Serum Glucose,Serum Potassium,Serum Sodium,Serum Uric Acid,Total Bilirubin,Total Protein,Urea Nitrogen,WBC
0,100891,BL,25.0,25.0,45.0,0.040000,2.37,0.080000,0.45000,160.000000,...,19.0,102.0,3.8,4.6,139.0,393.0,7.0,66.0,7.1,6.66000
1,101295,BL,26.0,23.0,61.0,0.040000,2.27,0.050000,0.41000,138.000000,...,25.0,105.0,9.2,4.2,142.0,208.0,7.0,62.0,5.0,4.95000
2,101482,BL,20.0,21.0,82.0,0.080000,2.40,0.090000,0.48000,160.000000,...,24.8,100.0,5.1,4.5,140.0,315.0,14.0,68.0,10.0,7.25000
3,101516,BL,20.0,26.0,73.0,0.060000,2.54,0.060000,0.42000,142.000000,...,21.9,101.0,5.6,4.3,138.0,345.0,15.0,79.0,9.3,5.39000
4,101749,BL,31.0,26.0,85.0,0.038054,2.50,0.125354,0.42644,142.776772,...,25.1,98.0,5.3,4.6,138.0,482.0,7.0,79.0,6.8,5.81322
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1328,75480,V04,23.0,13.0,86.0,0.080000,2.37,0.160000,0.46000,159.000000,...,20.5,101.0,8.3,4.2,138.0,291.0,19.0,78.0,8.9,9.24000
1329,75484,V04,18.0,23.0,45.0,0.060000,2.37,0.120000,0.48000,167.000000,...,22.1,101.0,4.7,4.5,143.0,256.0,14.0,73.0,4.3,4.62000
1330,75505,V04,7.0,12.0,91.0,0.030000,2.37,0.080000,0.37000,125.000000,...,23.3,102.0,4.7,4.2,139.0,315.0,5.0,65.0,6.1,4.89000
1331,75524,V04,27.0,23.0,77.0,0.020000,2.42,0.120000,0.42000,143.000000,...,23.2,100.0,4.7,3.9,138.0,297.0,7.0,69.0,5.7,4.28000


# Imaging

## DATscan

In [10]:
DAT_scan_info3=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Imaging/DaTSCAN/DATA/Xing_Core_Lab_-_Quant_SBR_12Sep2025.csv',dtype=str)
DAT_scan_info3=DAT_scan_info3.loc[DAT_scan_info3['PATNO'].isin(PATNOs)]
DAT_scan_info3=decoder_DF(DAT_scan_info3, code_rows, code_cols, module='DaTSCAN')
DAT_scan_info3=DAT_scan_info3.loc[DAT_scan_info3['Visit ID'].isin(orden_visitas)]
DAT_scan_info3['Visit ID'] = pd.Categorical(DAT_scan_info3['Visit ID'], categories=orden_visitas, ordered=True)
DAT_scan_info3=DAT_scan_info3.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)
DAT_scan_info3=DAT_scan_info3.loc[DAT_scan_info3['Was the scan able to be analyzed?']=='Yes',:]
DAT_scan_info3.drop(columns=['Identifier for the protocol','Previously Acquired Identifier','Date of Scan Acquisition','If DATSCAN_ANALYZED = No, the reason the scan was unable to be analyzed','If reason scan not analyzed is "OTHER", specify reason the scan was not analyzed','Was the scan able to be analyzed?'], inplace=True)
DAT_scan_info3.isna().sum()

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


PATNO                                                                                             0
Visit ID                                                                                          0
Radiopharmaceutical                                                                               0
Striatal Binding Ratio of the Whole Striatum referenced to Cerebral White Matter                  0
Striatal Binding Ratio of the Whole Caudate referenced to Cerebral White Matter                   0
Striatal Binding Ratio of the Whole Putamen referenced to Cerebral White Matter                   0
Striatal Binding Ratio of the Whole PreCaudate referenced to Cerebral White Matter                0
Striatal Binding Ratio of the Whole PosCaudate referenced to Cerebral White Matter                0
Striatal Binding Ratio of the Whole PreCommissural_Putamen referenced to Cerebral White Matter    0
Striatal Binding Ratio of the Whole PosCommissural_Putamen referenced to Cerebral White Matter    0


In [11]:
DAT_scan_info3

,PATNO,Visit ID,Radiopharmaceutical,Striatal Binding Ratio of the Whole Striatum referenced to Cerebral White Matter,Striatal Binding Ratio of the Whole Caudate referenced to Cerebral White Matter,Striatal Binding Ratio of the Whole Putamen referenced to Cerebral White Matter,Striatal Binding Ratio of the Whole PreCaudate referenced to Cerebral White Matter,Striatal Binding Ratio of the Whole PosCaudate referenced to Cerebral White Matter,Striatal Binding Ratio of the Whole PreCommissural_Putamen referenced to Cerebral White Matter,Striatal Binding Ratio of the Whole PosCommissural_Putamen referenced to Cerebral White Matter,...,Striatal Binding Ratio of the Right Caudate referenced to Cerebral White Matter,Striatal Binding Ratio of the Right Putamen referenced to Cerebral White Matter,Striatal Binding Ratio of the Right PreCaudate referenced to Cerebral White Matter,Striatal Binding Ratio of the Right PosCaudate referenced to Cerebral White Matter,Striatal Binding Ratio of the Right PreCommissural_Putamen referenced to Cerebral White Matter,Striatal Binding Ratio of the Right PosCommissural_Putamen referenced to Cerebral White Matter,Striatal Binding Ratio of the Right PreDorsalPutamen referenced to Cerebral White Matter,Striatal Binding Ratio of the Right PreVentralPutamen referenced to Cerebral White Matter,Striatal Binding Ratio of the Right PosDorsalPutamen referenced to Cerebral White Matter,Striatal Binding Ratio of the Right PosVentralPutamen referenced to Cerebral White Matter
0,100001,V04,123I-DaTscan,0.61,0.57,0.58,0.66,0.28,0.82,0.38,...,0.41,0.44,0.5,0.13,0.62,0.28,0.6,0.65,0.29,0.22
1,100001,V06,123I-DaTscan,0.55,0.54,0.51,0.65,0.23,0.71,0.35,...,0.5,0.29,0.58,0.24,0.4,0.19,0.35,0.47,0.16,0.26
2,100001,V10,123I-DaTscan,0.56,0.54,0.53,0.63,0.24,0.7,0.39,...,0.33,0.33,0.39,0.14,0.44,0.24,0.43,0.46,0.21,0.32
3,100002,V04,123I-DaTscan,0.43,0.38,0.41,0.43,0.23,0.54,0.31,...,0.4,0.39,0.42,0.33,0.5,0.29,0.44,0.57,0.32,0.2
4,100005,V04,123I-DaTscan,0.52,0.57,0.42,0.67,0.27,0.64,0.25,...,0.59,0.53,0.69,0.3,0.73,0.35,0.69,0.78,0.32,0.43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1597,75505,V06,123I-DaTscan,0.48,0.4,0.49,0.49,0.12,0.66,0.35,...,0.51,0.57,0.64,0.09,0.84,0.34,0.8,0.9,0.34,0.32
1598,75505,V10,123I-DaTscan,0.46,0.38,0.49,0.47,0.11,0.67,0.35,...,0.41,0.51,0.51,0.09,0.78,0.28,0.81,0.73,0.24,0.38
1599,75524,V06,123I-DaTscan,0.5,0.61,0.42,0.73,0.23,0.64,0.25,...,0.5,0.34,0.61,0.15,0.53,0.18,0.53,0.53,0.15,0.26
1600,75524,V10,123I-DaTscan,0.53,0.63,0.41,0.75,0.27,0.5,0.33,...,0.57,0.33,0.68,0.23,0.41,0.27,0.34,0.51,0.25,0.33


## MRI


In [12]:
mri_ad1=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Imaging/MR_Analysis_Results/DATA/FS7_APARC_CTH_12Sep2025.csv',dtype=str)
mri_ad1=mri_ad1.loc[mri_ad1['PATNO'].isin(PATNOs)]
mri_ad1=decoder_DF(mri_ad1, code_rows, code_cols, module='MRI_ANALYSIS')
mri_ad1=mri_ad1.loc[mri_ad1['Visit ID'].isin(orden_visitas)]
mri_ad1['Visit ID'] = pd.Categorical(mri_ad1['Visit ID'], categories=orden_visitas, ordered=True)
mri_ad1=mri_ad1.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)# solo hay BL

mri_ad1.columns = [
    col if i < 2 else 'TCH_' + col
    for i, col in enumerate(mri_ad1.columns)
]
mri_ad1



,PATNO,Visit ID,TCH_left bankssts thickness,TCH_left caudalanteriorcingulate thickness,TCH_left caudalmiddlefrontal thickness,TCH_left cuneus thickness,TCH_left entorhinal thickness,TCH_left fusiform thickness,TCH_left inferiorparietal thickness,TCH_left inferiortemporal thickness,...,TCH_right rostralmiddlefrontal thickness,TCH_right superiorfrontal thickness,TCH_right superiorparietal thickness,TCH_right superiortemporal thickness,TCH_right supramarginal thickness,TCH_right frontalpole thickness,TCH_right temporalpole thickness,TCH_right transversetemporal thickness,TCH_right insula thickness,TCH_right MeanThickness thickness
0,100001,BL,2.544,2.136,2.438,1.616,3.08,2.64,2.224,2.956,...,2.279,2.709,1.956,2.806,2.448,2.75,4.118,2.242,3.071,2.38038
1,100007,BL,2.518,2.442,2.426,1.773,3.361,2.707,2.336,2.745,...,2.325,2.537,2.047,2.725,2.391,2.787,3.668,2.132,3.093,2.40451
2,100012,BL,2.336,2.44,2.349,1.625,3.054,2.732,2.314,2.635,...,2.41,2.674,2.078,2.665,2.392,2.85,3.487,1.913,3.112,2.4158
3,100017,BL,2.706,2.141,2.304,1.603,3.239,2.754,2.151,2.998,...,2.217,2.416,1.974,2.592,2.438,2.606,3.73,2.038,3.041,2.33126
4,100018,BL,2.151,2.226,2.251,1.762,3.362,2.687,2.09,2.631,...,2.148,2.599,1.967,2.604,2.265,2.553,3.514,2.115,2.826,2.3076
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,75480,BL,2.491,2.14,2.226,1.793,3.438,2.419,2.128,2.611,...,2.017,2.296,1.687,2.523,2.001,2.827,3.682,1.947,2.62,2.16402
499,75484,BL,2.384,1.919,2.542,1.684,3.085,2.948,2.081,3.121,...,2.028,2.42,1.946,2.789,2.37,2.438,3.109,2.536,3.045,2.34383
500,75505,BL,2.265,2.172,2.337,1.549,3.213,2.433,2.147,2.562,...,2.113,2.547,1.992,2.681,2.39,2.717,3.349,2.138,2.766,2.23074
501,75524,BL,2.53,2.278,2.231,1.856,3.052,2.423,2.406,2.899,...,2.011,2.291,1.82,2.704,2.467,2.341,3.895,1.898,3.023,2.24775


In [13]:
mri_ad2=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Imaging/MR_Analysis_Results/DATA/FS7_APARC_SA_12Sep2025.csv',dtype=str)
mri_ad2=mri_ad2.loc[mri_ad2['PATNO'].isin(PATNOs)]
mri_ad2=decoder_DF(mri_ad2, code_rows, code_cols, module='MRI_ANALYSIS')
mri_ad2=mri_ad2.loc[mri_ad2['Visit ID'].isin(orden_visitas)]
mri_ad2['Visit ID'] = pd.Categorical(mri_ad2['Visit ID'], categories=orden_visitas, ordered=True)
mri_ad2=mri_ad2.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)# solo hay BL

mri_ad2.columns = [
    col if i < 2 else 'SA_' + col
    for i, col in enumerate(mri_ad2.columns)
]

mri_ad2

,PATNO,Visit ID,SA_left bankssts thickness,SA_left caudalanteriorcingulate thickness,SA_left caudalmiddlefrontal thickness,SA_left cuneus thickness,SA_left entorhinal thickness,SA_left fusiform thickness,SA_left inferiorparietal thickness,SA_left inferiortemporal thickness,...,SA_right rostralmiddlefrontal thickness,SA_right superiorfrontal thickness,SA_right superiorparietal thickness,SA_right superiortemporal thickness,SA_right supramarginal thickness,SA_right frontalpole thickness,SA_right temporalpole thickness,SA_right transversetemporal thickness,SA_right insula thickness,SA_right WhiteSurfArea area
0,100001,BL,952,737,2149,1477,567,3185,3916,3068,...,5548,6701,4929,3491,3190,405,425,331,2065,82306.6
1,100007,BL,1169,560,2266,1425,358,3059,4896,3582,...,5777,6979,4952,4165,3785,382,476,369,2338,90545.7
2,100012,BL,981,504,2068,1302,409,2944,4259,2985,...,4851,6406,4433,3540,3831,301,376,330,1967,77698.2
3,100017,BL,871,753,2374,1284,471,2987,4264,3629,...,5846,7608,5646,3602,3283,315,498,338,2447,85919
4,100018,BL,927,630,2312,1932,436,2786,3460,2909,...,5336,6287,5448,3211,3633,360,498,258,2207,81920.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,75480,BL,783,635,1911,1954,385,3130,4152,3321,...,5938,6151,5221,3347,3242,359,441,315,2196,83275
499,75484,BL,838,550,2419,1498,512,3622,5523,4283,...,6721,7752,5627,3627,3617,391,499,366,2181,92936.5
500,75505,BL,743,616,2330,1768,636,3549,4265,3798,...,5871,6867,4831,3298,2909,315,528,289,2228,85289.3
501,75524,BL,971,609,2200,1889,460,3834,3480,2658,...,6056,8013,5187,3659,3431,392,463,363,2243,86479.1


In [14]:
mri_ad3=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Imaging/MR_Analysis_Results/DATA/FS7_ASEG_VOL_12Sep2025.csv',dtype=str)
mri_ad3=mri_ad3.loc[mri_ad3['PATNO'].isin(PATNOs)]
mri_ad3=decoder_DF(mri_ad3, code_rows, code_cols, module='MRI_ANALYSIS')
mri_ad3=mri_ad3.loc[mri_ad3['Visit ID'].isin(orden_visitas)]
mri_ad3['Visit ID'] = pd.Categorical(mri_ad3['Visit ID'], categories=orden_visitas, ordered=True)
mri_ad3=mri_ad3.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)# solo hay BL
mri_ad3.columns = [
    col if i < 2 else 'VOL_' + col
    for i, col in enumerate(mri_ad3.columns)
]
mri_ad3

,PATNO,Visit ID,VOL_Left-WM-hypointensities,VOL_Brain-Stem,VOL_Left-non-WM-hypointensities,VOL_Optic-Chiasm,VOL_Right-WM-hypointensities,VOL_BrainSegVol,VOL_Right-Lateral-Ventricle,VOL_CC Central,...,VOL_Left-VentralDC,VOL_SupraTentorialVolNotVent,VOL_CC Mid Anterior,VOL_SupraTentorialVol,VOL_SubCortGrayVol,VOL_Right-Thalamus,VOL_Left-Lateral-Ventricle,VOL_CSF,VOL_SurfaceHoles,VOL_CerebralWhiteMatterVol
0,100001,BL,0,23182.4,0,93.8,0,1138230,8970.6,594.9,...,4147.7,975587,448,997071,58178,7695.4,7472.4,1170,7,482963
1,100007,BL,0,22783.9,0,129.6,0,1253700,19670.7,544.7,...,3826.1,1055260,574.9,1102160,56186,7565.4,19565.5,1081.9,11,517664
2,100012,BL,0,19230.2,0,121.9,0,1004910,10108.4,501,...,3490.3,868205,519.4,891193,50742,6456.6,8094,697.3,11,411422
3,100017,BL,0,21527.2,0,71.7,0,1160060,16843.3,410.3,...,3970.2,987213,506.2,1027310,62445,7936.6,17453.5,1053.5,14,483180
4,100018,BL,0,20615.3,0,173.7,0,1116420,24818.7,410.2,...,3866.8,931786,433.1,991345,49436,6025.8,24591.9,1432.6,4,472919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,75480,BL,0,18945.9,0,152.4,0,1088750,14886.6,361.4,...,3741.1,920712,357.1,960867,53304,6315.4,16004.5,1806.2,6,475926
499,75484,BL,0,21262.7,0,182,0,1291250,11234.9,411.8,...,3412.6,1122350,472.5,1156150,58233,7327.4,16253.5,1632.1,15,568643
500,75505,BL,0,21749.8,0,120.5,0,1190370,19812.3,418.1,...,3871.2,1002260,454.5,1049460,53567,6760.5,18822.6,1123,14,543459
501,75524,BL,0,22565.5,0,149.7,0,1141830,6467.9,687.3,...,4157.8,988305,515.5,1005240,56012,6695.4,6314.4,1261.9,18,502942


In [15]:
mri_ad4=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Imaging/MR_Analysis_Results/DATA/MRIQC_12Sep2025.csv',dtype=str)
mri_ad4=mri_ad4.loc[mri_ad4['PATNO'].isin(PATNOs)]
mri_ad4=decoder_DF(mri_ad4, code_rows, code_cols, module='MRI_ANALYSIS')
mri_ad4=mri_ad4.loc[mri_ad4['Visit ID'].isin(orden_visitas)]
mri_ad4['Visit ID'] = pd.Categorical(mri_ad4['Visit ID'], categories=orden_visitas, ordered=True)
mri_ad4=mri_ad4.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)# solo hay BL
mri_ad4.columns = [
    col if i < 2 else 'MRIQC_' + col
    for i, col in enumerate(mri_ad4.columns)
]
mri_ad4

,PATNO,Visit ID,MRIQC_coefficient of joint variation,MRIQC_contrast-to-noise ratio,MRIQC_entropy focus criterion,MRIQC_foreground-to-background energy ratio,MRIQC_relative partial volume of CSF,MRIQC_relative partial volume of GM,MRIQC_relative partial volume of WM,MRIQC_average full width at half maximum,...,MRIQC_intensity non-uniformity median,MRIQC_intensity non-uniformity range,MRIQC_signal-to-noise ratio of CSF,MRIQC_signal-to-noise ratio of GM,MRIQC_total signal-to-noise ratio,MRIQC_signal-to-noise ratio of WM,MRIQC_contrast-to-noise ratio of CSF,MRIQC_contrast-to-noise ratio of GM,MRIQC_total contrast-to-noise ratio,MRIQC_contrast-to-noise ratio of WM
0,100001,BL,0.75467,1.17342,0.532141,3915.71,7.87312,7.73193,7.8259,3.52686,...,0.770959,0.348713,1.71582,4.06404,4.66843,8.22543,15.7002,32.7787,30.8847,44.175
1,100007,BL,0.699713,1.22695,0.55907,5223.28,7.12244,6.9945,7.09615,3.81305,...,0.70034,0.315649,1.74526,3.79466,4.85913,9.03748,24.6581,52.3899,49.6697,71.9613
2,100012,BL,0.786624,1.2565,0.503256,7476.69,8.97034,8.82413,8.88364,4.16597,...,0.700651,0.287448,1.92908,5.18039,5.24376,8.62181,23.1593,53.6127,48.9829,70.1767
3,100017,BL,0.771662,1.11139,0.52647,2558.01,8.00773,7.85541,7.95244,3.62277,...,0.758359,0.354787,1.64778,3.89031,4.43141,7.75612,15.1884,32.0876,30.1488,43.1703
4,100018,BL,0.632073,1.25284,0.446879,12630.8,8.08693,7.95187,8.04872,3.84642,...,0.503439,0.145164,1.89025,3.90214,4.82565,8.68454,43.1844,92.7826,87.9314,127.827
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
509,75480,BL,0.635489,1.34124,0.486587,15930.5,9.56811,9.44044,9.54969,3.63175,...,0.583888,0.150495,1.97431,3.58526,4.55126,8.09419,13.8417,28.0086,27.4437,40.4808
510,75484,BL,0.44355,2.34077,0.568485,820.041,6.0206,5.98091,6.11592,3.98459,...,0.707743,0.174444,1.88195,2.82976,4.64185,9.21383,10.6566,22.4271,25.7484,44.1614
511,75505,BL,0.647624,1.50389,0.520639,11212.5,7.38571,7.29264,7.355,7.06983,...,0.309477,0.0567344,1.93207,4.97962,6.62631,12.9672,33.6504,79.3194,73.3594,107.108
512,75524,BL,0.721777,1.27299,0.410383,-1,10.7588,10.6296,10.7407,3.86573,...,0.788117,0.377452,1.60641,3.63397,4.37178,7.87495,17.3992,34.8462,33.9026,49.4623


In [16]:
mri_ad=mri_ad1.merge(mri_ad2, on=['PATNO', 'Visit ID'], how='inner')
mri_ad=mri_ad.merge(mri_ad3, on=['PATNO', 'Visit ID'], how='inner')
mri_ad=mri_ad.merge(mri_ad4, on=['PATNO', 'Visit ID'], how='inner')

# ANALYSIS DE PROGRESSION

In [17]:
progression_csv_upgrade(blood_data,prefix='BioSpecimen')
progression_multi_csv_upgrade(blood_data,prefix='BioSpecimen')
visit_csv_upgrade(blood_data,prefix='BioSpecimen')
blood_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_BioSpecimen_12SEP2025.csv', index=False)
cols_asignacion2(blood_data, df_secundario='BioSpecimen', df_main='IMAGING')

progression_csv_upgrade(mri_ad,prefix='MRI_ANALYSIS')
progression_multi_csv_upgrade(mri_ad,prefix='MRI_ANALYSIS')
visit_csv_upgrade(mri_ad,prefix='mri_ad')
mri_ad.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_MRI_ANALYSIS_12SEP2025.csv', index=False)
cols_asignacion2(mri_ad, df_secundario='MRI_ANALYSIS', df_main='IMAGING')

progression_csv_upgrade(DAT_scan_info3,prefix='DATSCAN')
progression_multi_csv_upgrade(DAT_scan_info3,prefix='DATSCAN')
visit_csv_upgrade(DAT_scan_info3,prefix='DATSCAN')
DAT_scan_info3.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_DATSCAN_12SEP2025.csv', index=False)
cols_asignacion2(DAT_scan_info3, df_secundario='DATSCAN', df_main='IMAGING')


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p1.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:65: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p2.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `

Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:254: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pBL.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pV04.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:265: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior,

Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:193: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p1.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p2.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:205: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, se

Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:277: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pV10.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:283: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pV12.fillna(False, inplace=True)


,df_main,df_secundario,df_secundario_col
0,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,PATNO
1,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Sporadic PD at Enrollment
2,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,RBD at Enrollment
3,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Pink1 Mutation at Enrollment
4,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Parkin Mutation at Enrollment
...,...,...,...
557,IMAGING,DATSCAN,Striatal Binding Ratio of the Right PosCommiss...
558,IMAGING,DATSCAN,Striatal Binding Ratio of the Right PreDorsalP...
559,IMAGING,DATSCAN,Striatal Binding Ratio of the Right PreVentral...
560,IMAGING,DATSCAN,Striatal Binding Ratio of the Right PosDorsalP...
